# 081 — Aceleradores, memoria y el límite real del cómputo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Codo A100 = 312 ÷ 2,039 ≈ **153 FLOP/byte**, frente a 295 de la
H100. La A100 tolera *mejor* la baja intensidad: su cómputo creció menos que su
ancho de banda, así que exige menos FLOP/byte para saturarse. Cada generación
nueva desplaza el codo hacia la derecha — ése es el muro de memoria hecho número.

**Ejercicio 2.** RTX 4090: 1 008 ÷ 4,6 ≈ **219 tok/s**; con MBU 70 % ≈ **153
tok/s**. M4 Max: 546 ÷ 4,6 ≈ **119 tok/s**; con MBU 70 % ≈ **83 tok/s**. La
razón entre ambos (1,85×) es exactamente la razón de anchos de banda: la
capacidad de cómputo no entra en la cuenta.

**Ejercicio 3.** FP16: `I = 2B/2 = B` → hace falta **B = 295**. Q4_K_M:
`I = 2B/0,57 ≈ 3,51·B` → **B ≈ 84**. Cuantizar mueve menos bytes por la misma
cantidad de FLOPs, así que sube la intensidad y se llega al codo con un lote 3,5×
menor: por eso cuantización y batching se refuerzan en vez de competir.

**Ejercicio 4.** Porque el techo sólo es útil si mides contra él. Sin MFU/MBU
exportados junto a TTFT y TPOT, "está lento" no se distingue de "está al 95 % del
límite físico", y se optimiza el código equivocado.


In [ ]:
# Ejercicio 1
codo = lambda tflops, tbs: tflops / tbs
print(f"codo A100 = {codo(312, 2.039):.1f} FLOP/byte  ·  H100 = {codo(989.5, 3.35):.1f}")

# Ejercicio 2
modelo_gb = 4.6
for nombre, bw in {"RTX 4090": 1008, "M4 Max": 546}.items():
    techo = bw / modelo_gb
    print(f"{nombre}: techo={techo:.0f} tok/s  con MBU 70% ≈ {techo*0.7:.0f} tok/s")

# Ejercicio 3
for etiqueta, b in {"FP16": 2.0, "Q4_K_M": 0.57}.items():
    print(f"{etiqueta}: lote para el codo = {295 * b / 2:.0f}")

# Ejercicio 4
result = run_lab("observability", seed=81)
assert result["kind"] == "observability" and result["seed"] == 81
assert result["evidence"] and result["limitations"]
show(result)


## Reflexión

1. Si tu servicio mide 480 tok/s y el techo teórico con lote 1 es 728, ¿qué
   optimizaciones dejan de tener sentido y cuáles siguen valiendo la pena?
2. Un proveedor anuncia 2× los TFLOPS de la H100 con el mismo ancho de banda.
   ¿Cuánto mejora tu decode con lote 1, y por qué?
3. ¿Por qué el prefill y el decode piden aceleradores con perfiles distintos, y
   qué implicaría separarlos en dos flotas?
